<a href="https://colab.research.google.com/github/Adyypower/Deep-learning-Models-or-topics/blob/main/FineTune_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes torch fastapi uvicorn

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!pip install datasets==3.2.0
!pip install -U bitsandbytes


In [ ]:
from datasets import load_dataset

dataset = load_dataset("liweili/c4_200m", split="train", streaming=True).take(10000)

for sample in dataset:
    print(sample)
    break


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'input': 'Bitcoin is for $7,094 this morning, which CoinDesk says.', 'output': 'Bitcoin goes for $7,094 this morning, according to CoinDesk.'}


In [ ]:
from transformers import AutoTokenizer , AutoModelForCausalLM

model_id = "microsoft/phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
def format_prompt(example):
    # The prompt structure the model will learn to complete
    prompt = f"""<|user|>
    Correct the grammar in this sentence: {example['input']}
    <|end|>
    <|assistant|>
    {example['output']}"""
    return prompt

def preprocess_function(examples):
    # Apply the prompt format and tokenize
    formatted_prompt = format_prompt(examples)
    return tokenizer(formatted_prompt, truncation=True, max_length=128)

# Apply the preprocessing to the entire dataset
processed_dataset = dataset.map(preprocess_function)

In [ ]:
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_8bit=True,
    device_map="auto",
    trust_remote_code=True
)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
from peft import LoraConfig , get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    # These are the correct modules for Phi-3
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(model,lora_config)



In [ ]:
!pip install -q -U transformers accelerate torch bitsandbytes

In [ ]:
import os
import transformers

# --- THE FIX ---
# Disable Weights & Biases logging (optional but avoids warnings)
os.environ["WANDB_DISABLED"] = "true"

# 🔑 Ensure caching is OFF to avoid 'DynamicCache' errors
peft_model.config.use_cache = False

# If you had gradient checkpointing enabled, keep it off for now
# peft_model.gradient_checkpointing_disable()

trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=processed_dataset,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        max_steps=250,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=20,
        output_dir="outputs",
        report_to="none",   # safeguard to avoid WANDB logging
        save_strategy="no", # optional: avoids eval/generation using cache
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(
        tokenizer, mlm=False
    ),
)

print("Starting training...")
trainer.train()
print("Training complete!")


Starting training...


Step,Training Loss
20,2.075700
40,1.901500
60,1.890200
80,1.872400
100,1.916700
120,1.905200
140,1.879900
160,1.920700
180,1.852300
200,1.894600


Training complete!
Starting training...


Step,Training Loss
20,1.805600
40,1.773000
60,1.765900
80,1.758000
100,1.811800
120,1.809400
140,1.806700
160,1.846800
180,1.805100
200,1.856000


Training complete!


In [ ]:
# This creates a single zip file from your model folder
!zip -r outputs.zip ./outputs

  adding: outputs/ (stored 0%)
  adding: outputs/runs/ (stored 0%)
  adding: outputs/runs/Sep15_07-00-09_d487175a2971/ (stored 0%)
  adding: outputs/runs/Sep15_07-00-09_d487175a2971/events.out.tfevents.1757919622.d487175a2971.3464.0 (deflated 62%)
  adding: outputs/runs/Sep15_07-04-21_d487175a2971/ (stored 0%)
  adding: outputs/runs/Sep15_07-04-21_d487175a2971/events.out.tfevents.1757919861.d487175a2971.3464.1 (deflated 62%)


In [ ]:
# This is the command that saves the usable model files
print("Saving the fine-tuned model adapter...")

adapter_output_dir = "phi3-grammar-corrector"
peft_model.save_pretrained(adapter_output_dir)

print(f"Model saved successfully! A new folder named '{adapter_output_dir}' has been created.")

Saving the fine-tuned model adapter...
Model saved successfully! A new folder named 'phi3-grammar-corrector' has been created.


In [ ]:
!ls -l

total 20
drwxr-xr-x 3 root root 4096 Sep 15 07:00 outputs
-rw-r--r-- 1 root root 5667 Sep 15 07:49 outputs.zip
drwxr-xr-x 2 root root 4096 Sep 15 07:52 phi3-grammar-corrector
drwxr-xr-x 1 root root 4096 Sep  9 13:46 sample_data


In [ ]:
!zip -r phi3-grammar-corrector.zip ./phi3-grammar-corrector

  adding: phi3-grammar-corrector/ (stored 0%)
  adding: phi3-grammar-corrector/README.md (deflated 65%)
  adding: phi3-grammar-corrector/adapter_config.json (deflated 55%)
  adding: phi3-grammar-corrector/adapter_model.safetensors (deflated 7%)


In [ ]:
from google.colab import files
files.download('phi3-grammar-corrector.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>